# 🇲🇦 Darija Translator on Google Colab (Qwen)

Runs the whole stack in one Colab VM:

1. **Qwen2.5** served by **Ollama** (OpenAI-compatible API on `localhost:11434`)
2. The **Jakarta REST** service (Jersey + Grizzly, Basic auth) on `localhost:8090`
3. A free **Cloudflare quick tunnel** giving you a public `https://….trycloudflare.com` URL for the Chrome extension, PHP, Python and React Native clients

**Before you start:** `Runtime ▸ Change runtime type ▸ T4 GPU` (a CPU runtime works only with a small model such as `qwen2.5:1.5b-instruct`). Then run the cells in order.

In [ ]:
#@title 1. Settings
MODEL      = "qwen2.5:7b-instruct" #@param ["qwen2.5:7b-instruct", "qwen2.5:3b-instruct", "qwen2.5:1.5b-instruct", "qwen2.5:14b-instruct"]
AUTH_USER  = "admin"               #@param {type:"string"}
AUTH_PASSWORD = "change-me-please" #@param {type:"string"}

In [ ]:
#@title 2. Install JDK 17, Maven and Ollama
import os, subprocess
!nvidia-smi -L || echo "No GPU detected: use a small model or enable a GPU runtime"
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless maven zstd > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version
!curl -fsSL https://ollama.com/install.sh | sh

GPU 0: Tesla T4 (UUID: GPU-f349cb59-c930-2898-3f29-4cfba387387c)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-24.04.1-Ubuntu, mixed mode, sharing)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
#@title 3. Start Ollama and download the Qwen model
import subprocess, time, urllib.request

ollama = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see ollama.log")
!ollama pull {MODEL}

In [ ]:
#@title 4. Get the project (upload darija-translator.zip)
import os, glob, shutil
from google.colab import files
os.chdir("/content")
shutil.rmtree("/content/darija-translator", ignore_errors=True)   # always start from the freshly uploaded zip
uploaded = files.upload()          # choose darija-translator.zip
zipname = next(iter(uploaded))
!unzip -q -o "{zipname}" -d /content
poms = glob.glob("/content/**/server/pom.xml", recursive=True)
assert poms, "server/pom.xml not found: upload the full darija-translator.zip"
PROJECT = os.path.dirname(os.path.dirname(poms[0]))
print("Project folder:", PROJECT)


Saving darija-translator (2).zip to darija-translator (2).zip
Project folder: /content/darija-translator


In [ ]:
#@title 5. Build the REST service with Maven
import subprocess, glob
os.environ.update({
    "LLM_BASE_URL": "http://localhost:11434/v1",
    "LLM_MODEL": MODEL,
    "AUTH_USER": AUTH_USER,
    "AUTH_PASSWORD": AUTH_PASSWORD,
    "PORT": "8090",
})
r = subprocess.run(["mvn", "-B", "-DskipTests", "package"], cwd=PROJECT + "/server",
                   capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])
JAR = next(iter(glob.glob(PROJECT + "/server/target/translator-service-*.jar")), None)
assert r.returncode == 0 and JAR, "Maven build failed: copy the output above and report it"
print("Built:", JAR)

mpress/1.25.0/commons-compress-1.25.0.jar (1.1 MB at 8.1 MB/s)
[INFO] Including org.glassfish.jersey.containers:jersey-container-grizzly2-http:jar:3.1.9 in the shaded jar.
[INFO] Including jakarta.inject:jakarta.inject-api:jar:2.0.1 in the shaded jar.
[INFO] Including org.glassfish.grizzly:grizzly-http-server:jar:4.0.2 in the shaded jar.
[INFO] Including org.glassfish.grizzly:grizzly-http:jar:4.0.2 in the shaded jar.
[INFO] Including org.glassfish.grizzly:grizzly-framework:jar:4.0.2 in the shaded jar.
[INFO] Including org.glassfish.jersey.core:jersey-common:jar:3.1.9 in the shaded jar.
[INFO] Including jakarta.annotation:jakarta.annotation-api:jar:2.1.1 in the shaded jar.
[INFO] Including org.glassfish.hk2:osgi-resource-locator:jar:1.0.3 in the shaded jar.
[INFO] Including org.glassfish.jersey.core:jersey-server:jar:3.1.9 in the shaded jar.
[INFO] Including org.glassfish.jersey.core:jersey-client:jar:3.1.9 in the shaded jar.
[INFO] Including jakarta.validation:jakarta.validation-api:ja

In [ ]:
#@title 6. Start the service and test it
import subprocess, time, json, urllib.request, urllib.error, base64
subprocess.run(["pkill", "-f", "translator-service"])  # stop any previous run
time.sleep(1)

server = subprocess.Popen(["java", "-jar", JAR],
                          stdout=open("server.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
last_error = None
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8090/api/translate/health", timeout=5); break
    except urllib.error.HTTPError as e:
        last_error = f"HTTP {e.code}: {e.read()[:500]!r}"; time.sleep(1)
    except Exception as e:
        last_error = repr(e); time.sleep(1)
else:
    print(open("server.log").read())
    raise RuntimeError("Service did not respond. Last error: " + str(last_error))

def translate(text, script="arabic", base="http://localhost:8090"):
    req = urllib.request.Request(base + "/api/translate",
        data=json.dumps({"text": text, "script": script}).encode(),
        headers={"Content-Type": "application/json",
                 "Authorization": "Basic " + base64.b64encode(f"{AUTH_USER}:{AUTH_PASSWORD}".encode()).decode()})
    return json.load(urllib.request.urlopen(req, timeout=300))

print(translate("Hello, how are you?"))
print(translate("Where is the train station?", "latin"))

{'original': 'Hello, how are you?', 'translation': 'هلا كيف داير؟', 'model': 'qwen2.5:7b-instruct'}
{'original': 'Where is the train station?', 'translation': 'kichbe el hadjeb?', 'model': 'qwen2.5:7b-instruct'}


In [ ]:
#@title 7. Expose it publicly (Cloudflare quick tunnel)
import re
!curl -fsSL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8090"],
                          stdout=subprocess.DEVNULL, stderr=open("tunnel.log", "w"))
PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m: PUBLIC_URL = m.group(0); break
    time.sleep(1)
assert PUBLIC_URL, open("tunnel.log").read()

print("Service URL for the clients:", PUBLIC_URL + "/api/translate")
print("Username:", AUTH_USER, "  Password:", AUTH_PASSWORD)
print("\nChrome extension: Server settings -> paste the URL above.")
print("Python : TRANSLATOR_URL=%s/api/translate TRANSLATOR_USER=%s TRANSLATOR_PASSWORD=... python translate_client.py \"Hello\"" % (PUBLIC_URL, AUTH_USER))
print("\nKeep this notebook running: the tunnel stops when the runtime stops.")

Service URL for the clients: https://xhtml-molecular-paint-books.trycloudflare.com/api/translate
Username: admin   Password: change-me-please

Chrome extension: Server settings -> paste the URL above.
Python : TRANSLATOR_URL=https://xhtml-molecular-paint-books.trycloudflare.com/api/translate TRANSLATOR_USER=admin TRANSLATOR_PASSWORD=... python translate_client.py "Hello"

Keep this notebook running: the tunnel stops when the runtime stops.


In [ ]:
#@title 8. Test through the public URL (optional)
print(translate("Good morning, I would like a coffee please.", base=PUBLIC_URL))

{'original': 'Good morning, I would like a coffee please.', 'translation': 'صباح الخير، بغيت قهوة بس.', 'model': 'qwen2.5:7b-instruct'}


In [ ]:
#@title 9. Stop everything
for p in (tunnel, server, ollama):
    p.terminate()